# Domain RAG Hybrid Search 실험

- Dense: `intfloat/multilingual-e5-large` (1024d, fastembed)
- Sparse: `Qdrant/bm25` (fastembed)
- Fusion: RRF (Reciprocal Rank Fusion)
- Collections: `domain_qna` (2,411개), `breed_meta` (1,125개)

## 0. 초기화

In [1]:
import os
import sys
sys.path.insert(0, "../../..")

from dotenv import load_dotenv
load_dotenv("../../../infra/.env")

from qdrant_client import QdrantClient
from qdrant_client.models import (
    FieldCondition,
    Filter,
    Fusion,
    FusionQuery,
    MatchAny,
    MatchValue,
    Prefetch,
    SparseVector,
)
from fastembed import SparseTextEmbedding, TextEmbedding

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
client = QdrantClient(url=QDRANT_URL)

for col in ["domain_qna", "breed_meta"]:
    info = client.get_collection(col)
    print(f"{col}: {info.points_count:,} points")

/opt/anaconda3/envs/final-project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


domain_qna: 2,411 points
breed_meta: 1,125 points


In [2]:
print("Dense 모델 로드 중...")
dense_model = TextEmbedding("intfloat/multilingual-e5-large")
print("Sparse 모델 로드 중...")
sparse_model = SparseTextEmbedding("Qdrant/bm25")
print("완료")

Dense 모델 로드 중...


/var/folders/n6/9fr973913k7931ymsg4ly6rc0000gn/T/ipykernel_42444/2718869934.py:2: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  dense_model = TextEmbedding("intfloat/multilingual-e5-large")


Sparse 모델 로드 중...
완료


In [3]:
def embed(query: str):
    """쿼리 → dense / sparse 벡터"""
    prefixed = f"query: {query}"
    dense_vec = list(dense_model.embed([prefixed]))[0].tolist()
    sparse_vec = list(sparse_model.embed([query]))[0]
    return dense_vec, SparseVector(
        indices=sparse_vec.indices.tolist(),
        values=sparse_vec.values.tolist(),
    )


def hybrid_search_qna(
    query: str,
    top_k: int = 10,
    species: str | None = None,      # dog / cat / both
    category: str | None = None,     # 건강 및 질병 / 사육 및 관리 / 영양 및 식단 / 행동 및 심리 / 여행 및 이동
):
    """domain_qna Hybrid Search (Dense + Sparse → RRF)"""
    dense_vec, sparse_vec = embed(query)

    must = []
    if species:
        must.append(FieldCondition(key="species", match=MatchAny(any=[species, "both"])))
    if category:
        must.append(FieldCondition(key="category", match=MatchValue(value=category)))
    qdrant_filter = Filter(must=must) if must else None

    results = client.query_points(
        collection_name="domain_qna",
        prefetch=[
            Prefetch(query=dense_vec, using="dense", limit=top_k * 3, filter=qdrant_filter),
            Prefetch(query=sparse_vec, using="sparse", limit=top_k * 3, filter=qdrant_filter),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )
    return results.points


def print_qna_results(points, query: str):
    print(f"\n쿼리: '{query}' — {len(points)}건")
    print("-" * 70)
    for i, p in enumerate(points, 1):
        pl = p.payload
        print(f"{i:2}. [{pl.get('species')}] [{pl.get('category')}] (출처: {pl.get('source')})  score={p.score:.4f}")
        # 원본 텍스트 재구성
        print(f"    Q: {pl.get('question', '-')}")
        answer = pl.get('answer', '-')
        print(f"    A: {answer[:100]}{'...' if len(str(answer)) > 100 else ''}")

## 1. domain_qna 기본 검색

In [22]:
results = hybrid_search_qna("강아지가 초콜릿을 먹었어요")
print_qna_results(results, "강아지가 초콜릿을 먹었어요")


쿼리: '강아지가 초콜릿을 먹었어요' — 10건
----------------------------------------------------------------------
 1. [dog] [행동 및 심리] (출처: bemypet)  score=0.5000
    Q: 강아지가 산책 중 멈춰 서서 '꼼짝도 하지 않는' 심리는?
    A: 주변 환경에 대한 공포, 체력 저하, 혹은 특정 장소에 대한 거부감의 표현입니다. 억지로 끌기보다는 강아지가 안심할 때까지 기다려주거나 간식으로 긍정적인 유도를 하는 것이 좋습니다...
 2. [dog] [건강 및 질병] (출처: manual)  score=0.5000
    Q: 강아지에게 초콜릿이 치명적인 이유는 무엇인가요?
    A: 초콜릿의 '테오브로민' 성분 때문입니다. 강아지는 이를 대사하는 능력이 매우 낮아 심박수 급증, 경련, 구토, 심지어 사망에 이르게 할 수 있습니다.
 3. [dog] [사육 및 관리] (출처: bemypet)  score=0.3690
    Q: 산책 중 강아지가 '길바닥의 오물'을 먹으려 할 때 대처법?
    A: 강아지가 먹기 전 보호자가 먼저 발견하고 '기다려' 혹은 '안돼' 지시를 내리는 것이 최선입니다. 입에 이미 물었다면 억지로 뺏기보다 더 맛있는 간식과 교환하는 방식을 사용해 소유...
 4. [dog] [영양 및 식단] (출처: bemypet)  score=0.3333
    Q: 강아지에게 '초콜릿'이 독이 되는 성분은 무엇인가요?
    A: 초콜릿의 '테오브로민' 성분은 강아지의 중추신경계와 심장에 독성을 일으킵니다. 다크 초콜릿일수록 함량이 높아 위험하며, 구토, 경련, 부정맥 등을 유발해 생명이 위험할 수 있습니다...
 5. [dog] [영양 및 식단] (출처: manual)  score=0.2500
    Q: 강아지에게 '초콜릿' 독성을 일으키는 테오브로민의 대사 기전은?
    A: 강아지는 테오브로민 대사 속도가 매우 느려 체내에 축적되며, 이는 

In [16]:
results = hybrid_search_qna("고양이 구토가 자주 일어나요")
print_qna_results(results, "고양이 구토가 자주 일어나요")


쿼리: '고양이 구토가 자주 일어나요' — 10건
----------------------------------------------------------------------
 1. [cat] [건강 및 질병] (출처: manual)  score=1.0000
    Q: 고양이 '신부전' 4기에서 구토가 잦아지는 요독증의 영향?
    A: 혈액 속에 배출되지 못한 노폐물(요소 등)이 위 점막을 자극하여 염증을 일으키고 뇌의 구토 중추를 자극하기 때문입니다.
 2. [cat] [건강 및 질병] (출처: manual)  score=0.5833
    Q: 고양이 '범백' 치료 시 '강제 급여'를 신중히 해야 하는 이유?
    A: 구토 증상이 심할 때 강제로 음식을 넣으면 흡인성 폐렴을 유발하거나 구토를 악화시켜 탈수를 심화시킬 수 있습니다. 수액 처치가 선행되어야 합니다.
 3. [cat] [건강 및 질병] (출처: bemypet)  score=0.3333
    Q: 고양이가 '노란 토'를 했다면 응급 상황인가요?
    A: 공복 시간이 너무 길어 위산이 역류했을 때 발생하는 경우가 많습니다. 한두 번의 구토 후 활력이 좋다면 식사 간격을 조절해 보되, 반복되거나 기력이 없다면 간이나 췌장 질환을 의심...
 4. [cat] [건강 및 질병] (출처: bemypet)  score=0.2526
    Q: 고양이가 사료를 씹지 않고 '그냥 삼키는 것'은 문제인가요?
    A: 고양이의 치아는 갈아내는 구조가 아닌 찢는 구조이므로 어느 정도 그냥 삼키는 것은 본능적입니다. 다만 구토가 잦다면 알갱이 크기가 작은 사료로 바꾸거나 슬로우 식기를 사용하여 속도...
 5. [dog] [건강 및 질병] (출처: bemypet)  score=0.2500
    Q: 강아지 '심장사상충' 예방약을 먹인 후 구토를 한다면?
    A: 약 성분에 대한 일시적인 거부 반응일 수 있습니다. 구토가 지속되면 성분을 바꾸거나(바르는 약 등) 수의사와 상담해야 합니다. 예방

## 2. species 필터 적용

In [6]:
# dog 필터 (both 포함)
results = hybrid_search_qna("슬개골 탈구 예방 방법", species="dog")
print_qna_results(results, "슬개골 탈구 예방 방법 (dog+both)")


쿼리: '슬개골 탈구 예방 방법 (dog+both)' — 10건
----------------------------------------------------------------------
 1. [dog] [건강 및 질병] (출처: manual)  score=1.0000
    Q: 슬개골 탈구'를 예방하기 위한 홈 케어법은?
    A: 발바닥 털을 짧게 깎아 미끄러짐을 방지하고, 소파나 침대에 전용 계단을 설치하며, 과체중이 되지 않도록 체중 관리를 철저히 해야 합니다.
 2. [dog] [건강 및 질병] (출처: bemypet)  score=0.5000
    Q: 강아지 '슬개골 탈구' 예방을 위한 실내 환경 조성법은?
    A: 미끄러운 바닥에 매트를 깔고, 침대나 소파 옆에 전용 계단을 설치하여 점프를 방지해야 합니다. 또한 과체중은 관절에 큰 무리를 주므로 적정 체중 유지가 필수입니다.
 3. [dog] [건강 및 질병] (출처: manual)  score=0.4167
    Q: 슬개골 탈구' 예방을 위한 '대퇴사두근' 강화 운동이 중요한 역학적 근거는?
    A: 대퇴사두근은 슬개골을 위쪽으로 당겨주는 주된 힘의 원천입니다. 이 근육이 탄탄하면 무릎뼈가 좌우로 흔들리는 것을 물리적으로 억제하여 탈구 빈도와 등급 진행을 늦출 수 있습니다.
 4. [dog] [건강 및 질병] (출처: manual)  score=0.4103
    Q: 슬개골 탈구' 2기와 3기를 구분하는 물리적 진단 기준은?
    A: 2기는 인위적으로 탈구시킨 후 다리를 펴면 제자리로 돌아가지만, 3기는 평소 탈구 상태이며 손으로 밀어야만 복구됩니다. 3기부터는 활차구 소실이 가속화됩니다.
 5. [dog] [건강 및 질병] (출처: manual)  score=0.3429
    Q: 슬개골 탈구' 수술 후 '경골 조면 이식술(TTT)'이 재발을 막는 물리적 원리는?
    A: 인대가 부착된 정강뼈 돌기(조면)를 외측으로 옮겨 고정함으로써, 안쪽으로 치우쳐 있던 허

In [7]:
# cat 필터
results = hybrid_search_qna("헤어볼 관리 어떻게 하나요", species="cat")
print_qna_results(results, "헤어볼 관리 어떻게 하나요 (cat+both)")


쿼리: '헤어볼 관리 어떻게 하나요 (cat+both)' — 10건
----------------------------------------------------------------------
 1. [cat] [사육 및 관리] (출처: manual)  score=0.6667
    Q: 털을 빗겨주는 그루밍이 헤어볼 예방 외에 주는 이점은?
    A: 피부의 혈액순환을 돕고 보호자와의 유대감을 강화합니다. 무엇보다 몸 구석구석을 살피며 혹이나 상처 등 건강 이상을 조기에 발견하는 기회가 됩니다.
 2. [cat] [건강 및 질병] (출처: bemypet)  score=0.5000
    Q: 사료 교체 후 고양이가 '설사'를 한다면 어떻게 해야 하나요?
    A: 급격한 식단 변화로 장내 미생물 균형이 깨진 경우입니다. 기존 사료와 새 사료를 9:1 비율부터 시작해 7~10일에 걸쳐 천천히 섞어주며 적응 기간을 가져야 합니다.
 3. [cat] [사육 및 관리] (출처: manual)  score=0.4762
    Q: 고양이 '털 공(헤어볼)' 배출을 돕는 캣그라스의 원리?
    A: 거친 풀의 섬유질이 위벽을 자극하여 구토를 유발하거나, 장운동을 촉진하여 대변으로 털이 함께 섞여 나오게 돕습니다.
 4. [cat] [사육 및 관리] (출처: bemypet)  score=0.3929
    Q: 고양이 털 빠짐을 줄이기 위한 가장 효과적인 관리법은?
    A: 매일 규칙적인 빗질로 죽은 털을 제거하는 것입니다. 이는 집안의 털 날림을 줄일 뿐만 아니라, 고양이가 털을 삼켜 생기는 '헤어볼' 문제 예방에도 매우 효과적입니다.
 5. [cat] [영양 및 식단] (출처: manual)  score=0.3833
    Q: 고양이 사료 성분 중 '셀룰로오스'가 헤어볼 관리에 좋은 이유?
    A: 불용성 식이섬유로서 장벽을 부드럽게 자극해 삼킨 털이 뭉치지 않고 변에 섞여 나오도록 밀어내는 빗질 효과를 제공합니다.
 6. [cat] [영양 및 식단

## 3. category 필터 적용

In [8]:
# 카테고리: 영양 및 식단
results = hybrid_search_qna(
    "단백질 얼마나 먹여야 하나요",
    species="dog",
    category="영양 및 식단",
)
print_qna_results(results, "단백질 얼마나 먹여야 하나요 (dog, 영양 및 식단)")


쿼리: '단백질 얼마나 먹여야 하나요 (dog, 영양 및 식단)' — 10건
----------------------------------------------------------------------
 1. [dog] [영양 및 식단] (출처: bemypet)  score=0.5909
    Q: 강아지에게 '달걀'을 줄 때 노른자만 줘야 하나요?
    A: 익힌 달걀은 흰자와 노른자 모두 훌륭한 단백질원입니다. 다만 날달걀 흰자의 '아비딘' 성분은 비오틴 흡수를 방해할 수 있으므로 가급적 완숙으로 익혀서 급여하세요.
 2. [dog] [영양 및 식단] (출처: manual)  score=0.5417
    Q: 신부전 처방식'에서 '단백질 함량'보다 '단백질의 질(Biological Value)'이 더 중요한 이유는?
    A: 신장이 노폐물을 거르지 못하므로, 암모니아 발생이 적고 몸에 100% 흡수되는 양질의 단백질을 최소량만 주어야 하기 때문입니다. 질 낮은 단백질은 독소만 생성하여 신장의 파괴를 가...
 3. [dog] [영양 및 식단] (출처: manual)  score=0.3768
    Q: 노령견 사료에서 단백질 질(Quality)은 높이고 양(Quantity)은 조절해야 하는 이유는?
    A: 신장 기능이 저하된 경우 과도한 단백질은 질소 노폐물을 늘려 신장에 부담을 주지만, 근감소증 예방을 위해 흡수율이 높은 양질의 단백질은 반드시 필요하기 때문입니다.
 4. [dog] [영양 및 식단] (출처: manual)  score=0.3333
    Q: 단백질 제한 식단'이 간문맥 전신 단락(PSS) 환견의 암모니아 수치를 낮추는 원리는?
    A: 간을 거치지 않고 혈류로 직접 들어가는 암모니아는 주로 단백질 대사 과정에서 생성됩니다. 단백질 양을 조절하되 흡수율을 높여 간성 뇌증(Hepatic Encephalopathy) ...
 5. [dog] [영양 및 식단] (출처: bemypet)  score=0.3333
    Q: 

In [9]:
# 카테고리: 건강 및 질병
results = hybrid_search_qna(
    "신장 수치가 높게 나왔어요",
    species="cat",
    category="건강 및 질병",
)
print_qna_results(results, "신장 수치가 높게 나왔어요 (cat, 건강 및 질병)")


쿼리: '신장 수치가 높게 나왔어요 (cat, 건강 및 질병)' — 10건
----------------------------------------------------------------------
 1. [cat] [건강 및 질병] (출처: manual)  score=0.8333
    Q: 고양이 '갑상선 기능 항진증'이 신부전을 가리는 이유는?
    A: 과도한 호르몬이 심박수를 높여 신장 혈류량을 강제로 늘리기 때문입니다. 치료를 시작하면 숨겨져 있던 신부전 수치가 수면 위로 드러나는 경우가 많습니다.
 2. [cat] [건강 및 질병] (출처: manual)  score=0.5000
    Q: 고양이 '비대성 심근증(HCM)' 진단에 'proBNP' 키트 검사가 유용한가요?
    A: 네. 심장 근육이 늘어나거나 압박받을 때 방출되는 단백질 수치를 측정하여, 심장병의 가능성을 빠르게 선별(Screening)할 수 있는 매우 유용한 1차 검사입니다.
 3. [cat] [건강 및 질병] (출처: manual)  score=0.4500
    Q: 고양이 '만성 신부전' 환묘의 혈액 검사상 'SDMA' 수치가 중요한 이유?
    A: 기존의 '번(BUN)'이나 '크레아티닌'보다 신장 손상을 25~40% 더 빨리 감지할 수 있는 조기 진단 지표이기 때문입니다. 신장 질환 예방의 핵심 수치입니다.
 4. [cat] [건강 및 질병] (출처: manual)  score=0.4000
    Q: 고양이 '만성 신부전' 환묘의 잇몸에서 피가 나고 악취가 난다면?
    A: 높은 요독 수치로 인해 구강 점막이 괴사하는 '요독성 구내염'입니다. 단순 치과 질환이 아니므로 신장 수치 관리를 통한 요독 제거가 최우선입니다.
 5. [cat] [건강 및 질병] (출처: manual)  score=0.2917
    Q: 고양이 '갑상선 기능 항진증'이 신부전 증상을 가릴 수 있나요?
    A: 네. 항진증으로 인해 높아진 혈압이 신장의 여과율을 일시적으로 높

## 4. Dense only vs Sparse only vs Hybrid 비교

In [10]:
QUERY = "강아지 분리불안 훈련 방법"
TOP_K = 5
dense_vec, sparse_vec = embed(QUERY)

must = [FieldCondition(key="species", match=MatchAny(any=["dog", "both"]))]
f = Filter(must=must)

dense_results = client.query_points(
    collection_name="domain_qna", query=dense_vec, using="dense",
    limit=TOP_K, with_payload=True, query_filter=f,
).points

sparse_results = client.query_points(
    collection_name="domain_qna", query=sparse_vec, using="sparse",
    limit=TOP_K, with_payload=True, query_filter=f,
).points

hybrid_results = hybrid_search_qna(QUERY, top_k=TOP_K, species="dog")

def show(label, points):
    print(f"\n{'='*10} {label} {'='*10}")
    for i, p in enumerate(points, 1):
        pl = p.payload
        print(f"{i}. [{pl.get('category')}] {pl.get('question', '-')[:60]} | score={p.score:.4f}")

show("Dense only", dense_results)
show("Sparse only", sparse_results)
show("Hybrid (RRF)", hybrid_results)


========== Dense only ==========
1. [행동 및 심리] 분리불안' 환견에게 '출근 준비 루틴'을 무작위화(Randomization)하는 것이 효과적인 이유는? | score=0.8712
2. [사육 및 관리] 포식적 위안(Predatory Comfort)' 활동이 강아지의 분리불안 완화에 도움을 주는 법은? | score=0.8666
3. [사육 및 관리] 차별 강화(Differential Reinforcement)' 기법을 활용하여 '짖는 개'를 교육하는 원리는 | score=0.8643
4. [사육 및 관리] 포식적 위안' 활동이 강아지의 분리불안 완화에 실질적 도움을 주는 원리는? | score=0.8636
5. [사육 및 관리] 다견 가정에서 '싸움' 예방을 위해 '공간의 수직적 분리'를 활용하는 방법은? | score=0.8608

========== Sparse only ==========
1. [행동 및 심리] 강아지가 산책 중 '다른 개를 보고 짖지 않게' 하는 가장 빠른 방법? | score=25.0991
2. [사육 및 관리] 강아지 '스카프형 케이프' 착용 시 고정 방법? | score=18.6887
3. [사육 및 관리] 리콜(Call)' 훈련 시 강아지가 왔을 때 바로 리드줄을 채우면 안 되는 학습적 이유는? | score=16.6589
4. [사육 및 관리] 강아지 '켄넬 교육(크레이트 훈련)'이 필요한 이유는? | score=16.5426
5. [사육 및 관리] 포식적 위안' 활동이 강아지의 분리불안 완화에 실질적 도움을 주는 원리는? | score=15.3660

========== Hybrid (RRF) ==========
1. [행동 및 심리] 강아지가 산책 중 '다른 개를 보고 짖지 않게' 하는 가장 빠른 방법? | score=0.6429
2. [행동 및 심리] 분리불안' 환견에게 '출근 준비 루틴'을 무작위화(Randomization)하는 것이 효과적인 이유는? | score=0.6111
3. [

## 5. breed_meta payload filter 조회

In [11]:
def get_breed_meta(
    breed_name: str,
    species: str,
    age_group: str | None = None,
):
    """
    breed_meta payload filter 조회 (Hybrid Search 미사용)
    - breed_name + species exact match
    - age_group 지정 시 추가 필터
    """
    must = [
        FieldCondition(key="breed_name", match=MatchValue(value=breed_name)),
        FieldCondition(key="species", match=MatchValue(value=species)),
    ]
    if age_group:
        must.append(FieldCondition(key="age_group", match=MatchValue(value=age_group)))

    results = client.scroll(
        collection_name="breed_meta",
        scroll_filter=Filter(must=must),
        with_payload=True,
        limit=10,
    )
    return results[0]


def print_breed_results(points):
    for p in points:
        pl = p.payload
        print(f"[{pl.get('species')}] {pl.get('breed_name')} ({pl.get('breed_name_en')}) / {pl.get('age_group')}")
        print(f"  그룹: {pl.get('group')} | 체급: {pl.get('size_class')}")
        print(f"  좋아하는 사료: {pl.get('preferred_food')}")
        print(f"  건강제품: {pl.get('health_products')}")
        print()

In [12]:
# 말티즈 어덜트
points = get_breed_meta("말티즈", "dog", "어덜트")
print_breed_results(points)

[dog] 말티즈 (Maltese) / 어덜트
  그룹: 장모종 | 체급: ['XS']
  좋아하는 사료: 초소형견 전용 어덜트 사료
  건강제품: 눈 영양제



In [13]:
# 말티즈 전 연령대
points = get_breed_meta("말티즈", "dog")
print_breed_results(points)

[dog] 말티즈 (Maltese) / 퍼피
  그룹: 장모종 | 체급: ['XS']
  좋아하는 사료: 초소형견 전용 퍼피 사료
  건강제품: 눈 영양제

[dog] 말티즈 (Maltese) / 어덜트
  그룹: 장모종 | 체급: ['XS']
  좋아하는 사료: 초소형견 전용 어덜트 사료
  건강제품: 눈 영양제

[dog] 말티즈 (Maltese) / 시니어
  그룹: 장모종 | 체급: ['XS']
  좋아하는 사료: 초소형견 전용 시니어 사료
  건강제품: 눈 영양제



In [14]:
# 코리안 숏헤어 키튼
points = get_breed_meta("코리안 숏헤어", "cat", "키튼")
print_breed_results(points)

[cat] 코리안 숏헤어 (Korean Short Hair) / 키튼
  그룹: 단모종 | 체급: ['M']
  좋아하는 사료: 기호성 좋은 전연령 사료
  건강제품: 구강 유산균



## 6. category 분포 확인

In [15]:
import pandas as pd

all_points = []
offset = None
while True:
    batch, offset = client.scroll(
        collection_name="domain_qna",
        with_payload=True,
        limit=500,
        offset=offset,
    )
    all_points.extend(batch)
    if offset is None:
        break

df = pd.DataFrame([p.payload for p in all_points])
print("species 분포")
print(df["species"].value_counts())
print("\ncategory 분포")
print(df["category"].value_counts())
print("\nsource 분포")
print(df["source"].value_counts())

species 분포
species
dog     1129
cat     1115
both     167
Name: count, dtype: int64

category 분포
category
사육 및 관리    710
건강 및 질병    576
영양 및 식단    563
행동 및 심리    554
여행 및 이동      8
Name: count, dtype: int64

source 분포
source
manual     1999
bemypet     361
biteme       51
Name: count, dtype: int64
